In [1]:
import re
import numpy as np
import pandas as pd

In [2]:
input_path = "../data/processed/extracted/2020.csv"
output_path = "../data/processed/2020_cleaned.csv"

In [3]:
# 1. Load 2020 dataset and strip HTML tags / non-breaking spaces from column names
df = pd.read_csv(input_path)
df.columns = [re.sub(r"<[^>]+>", "", col).replace("\xa0", " ").strip() for col in df.columns]

In [4]:
# 2. Drop exact duplicates and filter valid ages (14–100)
df = df.drop_duplicates().copy()
age_col = "What is your age?"
df = df[df[age_col].between(14, 100)].copy()
df[age_col] = df[age_col].astype(int)

In [5]:
# 3. Parse timestamp columns
for dt_col in ["Start Date (UTC)", "Submit Date (UTC)"]:
    if dt_col in df.columns:
        df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")

In [6]:
# 4. Clean free-text gender responses
gender_col = "What is your gender?"
male_vals = {"male", "m", "man", "mail", "cisgender male"}
female_vals = {"female", "f", "woman", "female, she/her"}
non_gender_vals = set()

def clean_gender_2020(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in male_vals:
        return "male"
    if s in female_vals:
        return "female"
    if s in non_gender_vals:
        return np.nan
    return "other"

df[gender_col] = df[gender_col].map(clean_gender_2020)

In [7]:
# 5. Convert 0/1 binary columns to Yes/No and trim whitespace on object columns
binary_cols = [
    "Are you self-employed?",
    "Is your employer primarily a tech company/organization?",
    "Is your primary role within your company related to tech/IT?",
    "Do you have previous employers?",
    "Have you ever been diagnosed with a mental health disorder?",
    "Have you had a mental health disorder in the past?",
    "Have you ever sought treatment for a mental health disorder from a mental health professional?",
    "Are you openly identified at work as a person with a mental health issue?",
]
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].replace({0: "No", 1: "Yes", 0.0: "No", 1.0: "Yes"})

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

C:\Users\chami\AppData\Local\Temp\ipykernel_2508\3443275956.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [8]:
# 6. Make company size an ordered categorical column and add numeric indicator (1-6)
ORDER_EMP = ["1-5", "6-25", "26-100", "100-500", "500-1000", "More than 1000"]
emp_code_map = {
    "1-5": 1,
    "6-25": 2,
    "26-100": 3,
    "100-500": 4,
    "500-1000": 5,
    "More than 1000": 6,
}

emp_col = "How many employees does your company or organization have?"
df[emp_col] = pd.Categorical(df[emp_col], categories=ORDER_EMP, ordered=True)
df["no_employees_code"] = df[emp_col].map(emp_code_map).astype("Int64")

In [9]:
# 7. Save cleaned 2020 dataset
df = df.reset_index(drop=True)
df.to_csv(output_path, index=False)

print("Saved:", output_path, "| Shape:", df.shape)
print(df[[emp_col, "no_employees_code"]].value_counts(dropna=False).sort_index())

Saved: ../data/processed/2020_cleaned.csv | Shape: (178, 121)
How many employees does your company or organization have?  no_employees_code
1-5                                                         1                     1
                                                            2                     0
                                                            3                     0
                                                            4                     0
                                                            5                     0
                                                            6                     0
                                                            <NA>                  0
6-25                                                        1                     0
                                                            2                    10
                                                            3                     0
                    

In [11]:
print(df.head(5))

                                  #  *Are you self-employed?*  \
0  zwrffw6ykfo82ft1twvzwrffw6c6wsfv                         1   
1  zhdmhaa8r0125c4zmoi7qzhdmtjrakhm                         1   
2  x4itwa9hnlw7qke4y5xibx4itwa9yzl5                         1   
3  x3v3oimu5pn0043n8x3v3oizaybhwwto                         1   
4  uyp6re7bhnyx6gez09uyp6re72z0e4e4                         1   

  How many employees does your company or organization have?  \
0                                                NaN           
1                                                NaN           
2                                                NaN           
3                                                NaN           
4                                                NaN           

  Is your employer primarily a tech company/organization?  \
0                                                NaN        
1                                                NaN        
2                                        